# DOTA-v1.0 Ship + YOLOv8n-OBB
从 ModelScope 下载 DOTA-v1.0，生成 1024 像素船舶单类子集，并使用 Colab GPU 训练。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q -o \
  "/content/drive/MyDrive/OrientedDetection_CAM_colab.zip" \
  -d "/content/drive/MyDrive"

PROJECT_DIR = "/content/drive/MyDrive/OrientedDetection_CAM"
%cd $PROJECT_DIR

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/OrientedDetection_CAM


In [2]:
!pip install -q -r requirements-ml.txt -r requirements-data.txt
import torch
assert torch.cuda.is_available(), '请在 Colab 运行时中选择 GPU'
print(torch.cuda.get_device_name(0))

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
google-adk 2.3.0 requires fastapi<1,>=0.133, but you have fastapi 0.116.1 which is incompatible.
google-adk 2.3.0 requires starlette<2,>=1.0.1, but you have starlette 0.47.3 which is incompatible.
gradio 6.19.0 requires starlette<2.0,>=1.0.1, but you have starlette 0.47.3 which is incompatible.
python-fasthtml 0.14.3 requires starlette>=1.0.1, but you have starlette 0.47.3 which is incompatible.
Tesla T4


## 下载与准备 DOTA Ship 子集
缓存和切片放在 Colab 临时磁盘，避免大量小文件拖慢 Google Drive。最终权重与曲线会复制回项目。

In [3]:
!python scripts/download_dota_modelscope.py --cache-dir /content/modelscope-cache

2026-07-14 03:12:53,216 - modelscope - ERROR - [MS_DOWNLOAD] _request_with_retry_ms ReadTimeout: method=HEAD, url=https://modelscope.cn/api/v1/datasets/yolo_master/DOTAv1/repo?Source=SDK&Revision=master&FilePath=README.md&View=False, timeout=100, error=HTTPSConnectionPool(host='modelscope.cn', port=443): Read timed out. (read timeout=100)
2026-07-14 03:12:53,216 - modelscope - INFO - HEAD request to https://modelscope.cn/api/v1/datasets/yolo_master/DOTAv1/repo?Source=SDK&Revision=master&FilePath=README.md&View=False timed out, retrying... [0.3333333333333333]
2026-07-14 03:12:55,656 - modelscope - INFO - [MS_DOWNLOAD] get_from_cache_ms downloading: url=https://modelscope.cn/api/v1/datasets/yolo_master/DOTAv1/repo?Source=SDK&Revision=master&FilePath=README.md&View=False
2026-07-14 03:16:54,551 - modelscope - INFO - storing https://modelscope.cn/api/v1/datasets/yolo_master/DOTAv1/repo?Source=SDK&Revision=master&FilePath=README.md&View=False in cache at /content/modelscope-cache/datasets/

In [6]:
from pathlib import Path
from modelscope.hub.snapshot_download import dataset_snapshot_download

CACHE_DIR = Path("/content/modelscope-cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

dataset_snapshot_download(
    dataset_id="yolo_master/DOTAv1",
    local_dir=str(CACHE_DIR),
    allow_file_pattern="datasets/yolo_master/DOTAv1.zip",
    max_workers=4,
)

archive = CACHE_DIR / "datasets/yolo_master/DOTAv1.zip"

assert archive.exists(), f"下载完成但未找到文件：{archive}"
print("压缩包：", archive)
print("大小：", archive.stat().st_size, "bytes")

2026-07-14 03:43:47,060 | INFO    | modelscope_hub.download | Downloading 1 files from yolo_master/DOTAv1@master


Downloading:   0%|          | 0/1 [00:00<?, ?file/s]

DOTAv1.zip:   0%|          | 0.00/1.99G [00:00<?, ?B/s]

压缩包： /content/modelscope-cache/datasets/yolo_master/DOTAv1.zip
大小： 1988510024 bytes


In [7]:
!python scripts/prepare_dota_ship.py "{archive}" \
  --output /content/dota_ship \
  --work-dir /content/dota_work \
  --crop-size 1024 \
  --gap 200 \
  --negative-ratio 0.25 \
  --max-train 2500 \
  --max-val 600

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
val: 100% 458/458 [00:54<00:00,  8.35it/s]
train: 100% 1411/1411 [02:39<00:00,  8.86it/s]
DOTA ship subset: /content/dota_ship
train: 1869 positive patches, 467 negatives, 57616 ships
val: 480 positive patches, 120 negatives, 14152 ships


In [8]:
import json
from pathlib import Path

manifest_path = Path("/content/dota_ship/manifest.json")
assert manifest_path.exists(), manifest_path

manifest = json.loads(manifest_path.read_text())
manifest

{'source_root': '/content/modelscope-cache/datasets/yolo_master/DOTAv1.zip',
 'prepared_root': '/content/dota_work/dota_patches',
 'output_root': '/content/dota_ship',
 'ship_class_id': 1,
 'crop_size': 1024,
 'gap': 200,
 'splits': {'train': {'positives': 1869, 'negatives': 467, 'objects': 57616},
  'val': {'positives': 480, 'negatives': 120, 'objects': 14152}}}

In [9]:
!python -m detection.train --data /content/dota_ship/dota_ship.yaml --model yolov8n-obb.pt --epochs 80 --imgsz 640 --batch 16 --device 0 --workers 2 --project /content/runs/dota_ship --name yolov8n-obb

100% 6.26M/6.26M [00:00<00:00, 103MB/s]
New https://pypi.org/project/ultralytics/8.4.95 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.169 🚀 Python-3.12.13 torch-2.7.1+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dota_ship/dota_ship.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8

In [11]:
import shutil
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/OrientedDetection_CAM")

# 1. 保存最佳权重
best = Path("/content/runs/dota_ship/yolov8n-obb/weights/best.pt")
assert best.exists(), best

weights_dir = PROJECT_DIR / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(best, weights_dir / "best.pt")

# 2. 保存完整训练结果
run_source = Path("/content/runs/dota_ship/yolov8n-obb")
drive_run = PROJECT_DIR / "runs/dota_ship/yolov8n-obb-colab"
drive_run.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(run_source, drive_run, dirs_exist_ok=True)

# 3. 获取或重新生成数据子集压缩包
subset_archive = Path("/content/dota_ship-subset.zip")

if not subset_archive.exists():
    assert Path("/content/dota_ship").exists(), "DOTA Ship 子集临时目录已不存在"
    subset_archive = Path(
        shutil.make_archive(
            "/content/dota_ship-subset",
            "zip",
            root_dir="/content/dota_ship",
        )
    )

# 4. 创建之前缺失的目标目录
datasets_dir = PROJECT_DIR / "datasets"
datasets_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(subset_archive, datasets_dir / "dota_ship-subset.zip")

print("权重：", weights_dir / "best.pt")
print("训练结果：", drive_run)
print("数据子集：", datasets_dir / "dota_ship-subset.zip")

权重： /content/drive/MyDrive/OrientedDetection_CAM/weights/best.pt
训练结果： /content/drive/MyDrive/OrientedDetection_CAM/runs/dota_ship/yolov8n-obb-colab
数据子集： /content/drive/MyDrive/OrientedDetection_CAM/datasets/dota_ship-subset.zip


In [12]:
!python -m detection.evaluate --weights weights/best.pt --data /content/dota_ship/dota_ship.yaml --split val --device 0 --output artifacts/colab-evaluation.json

Ultralytics 8.3.169 🚀 Python-3.12.13 torch-2.7.1+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-obb summary (fused): 81 layers, 3,077,414 parameters, 0 gradients, 8.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2206.3±1476.7 MB/s, size: 203.7 KB)
val: Scanning /content/dota_ship/labels/val.cache... 600 images, 120 backgrounds, 4 corrupt: 100% 600/600 [00:00<?, ?it/s]
val: /content/dota_ship/images/val/P0259__1024__824___0.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.085937 1.090821]
val: /content/dota_ship/images/val/P0259__1024__824___1245.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.041014]
val: /content/dota_ship/images/val/P0261__1024__390___0.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.179688]
val: /content/dota_ship/images/val/P0261__1024__390___74.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.179688]
                 Class   